# 01 · Reading inquiry emails: what is there, and what is missing

This notebook covers the first part of the swimlane diagram:
**scrape data from the email → match it against the quote template → flag missing or faulty data → checkpoint: is the data sufficient?**

```
email ──▶ extractor ──▶ checks (plain code) ──▶ issues + draft reply to the client
          (LLM or rules)   • is the evidence really in the email?
                           • fill YC default terms, marked as assumptions
                           • what blocks the quote, what only needs a look
```

Two extractors are compared:
1. **Baseline (rules):** regular expressions written by looking at the three sample emails. No AI.
2. **LLM:** a language model reads the email and fills in the template, including *missing* and *needs review*.

Both are scored against a hand-made answer key (`data/test_cases/`), where `null` means "not in the email".
A good extractor must catch the values that are there **and** report the gaps instead of guessing.

The engine code lives in `quotetool/`; this notebook only calls it.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display, Markdown
import quotetool as qt
from quotetool import llm, report, evaluate

pd.set_option("display.max_colwidth", 70)
INBOX, KEYS, CACHE = ROOT / "data/inbox", ROOT / "data/test_cases", ROOT / "data/lake/processed"

inbox = qt.load_inbox(INBOX)
print(f"{len(inbox)} emails:", ", ".join(inbox))

## Step 1 · The input

One example as the tool receives it: header lines, the attachment names, then the body.
This one is hard: a table whose description column wraps across lines, a unit trap (one SET = 2 pieces)
hidden in brackets, and a long material paragraph that is background, not order data.

In [ ]:
print(inbox["case02_customer_P"].text)

## Step 2 · The LLM reads the emails

**Setup on your laptop (once):**
1. Install Ollama from https://ollama.com
2. In a terminal: `ollama pull qwen3.5:9b` (Qwen handles Chinese well). On a laptop with 8 GB RAM use `qwen3.5:4b`.
3. Ollama runs in the background; re-run this notebook.

Nothing is sent to the internet: the model runs on your machine, which respects YC's red line on drawings and prices.

Replies are cached in `data/lake/processed/`, so after one run the notebook works without the model.
Delete a cache file (or pass `refresh=True`) to ask the model again.

**Comparing models:** put every model you want to compare in `MODELS`. Each one is scored in Step 5.
A model you have not pulled is skipped, unless its saved answers are already in `data/lake/processed/`.
The first model in the list is the one shown in Step 4.

In [ ]:
MODELS = ["qwen3.5:9b", "qwen2.5:7b"]   # newest first; add or remove models here

for model in MODELS:
    ok, message = llm.check(qt.LLMConfig(model=model))
    print(("✅ " if ok else "⚠️ ") + message)

In [ ]:
llm_results = {}   # model -> {case_id: filled template}
for model in MODELS:
    config = qt.LLMConfig(model=model)
    answers = {}
    for case_id, email in inbox.items():
        try:
            answers[case_id] = qt.extract_llm(email, config, cache_dir=CACHE)
            print(f"{model} · {case_id}: {len(answers[case_id].items)} lines extracted")
        except Exception as e:  # model not running, or reply not parseable
            print(f"{model} · {case_id}: skipped ({type(e).__name__}: {str(e)[:120]})")
    if len(answers) == len(inbox):
        llm_results[model] = answers

## Step 3 · The baseline: rules only

Runs anywhere, no model needed. The rules were written with these three emails in front of us, so treat its
score as a best case for rules: a new customer with a different layout will break them.

In [ ]:
baseline_results = {case_id: qt.extract_baseline(email) for case_id, email in inbox.items()}
for case_id, ex in baseline_results.items():
    print(f"{case_id}: {len(ex.items)} lines extracted")

## Step 4 · Checkpoint: is the data sufficient to quote?

The checks are plain code, so they behave the same every time and are easy to change:

| Severity | Meaning | Example |
|---|---|---|
| **blocking** | Cannot quote without it: part number, quantity, material, drawing | Quantities not given |
| **check** | A person should look: ambiguous value, or evidence not found in the email | 144 SET × 2 = 288 pieces? |
| **assumption** | Filled with YC's usual terms and shown on the quote | No incoterm → FOB Shanghai |
| **info** | Worth knowing, nothing to do | No deadline given |

If anything is blocking, the tool drafts a reply asking the client. A person edits and sends it.

The results below use the first model in `MODELS` that read all emails, otherwise the baseline.

In [ ]:
shown = next((m for m in MODELS if m in llm_results), None)
results = llm_results[shown] if shown else baseline_results
print("Showing results from:", f"LLM ({shown})" if shown else "baseline (rules); no LLM results available")
checked = {case_id: qt.check(results[case_id], inbox[case_id]) for case_id in inbox}

def show(case_id):
    r = checked[case_id]
    verdict = "✅ ready to quote" if r.ready_to_quote else "❌ not ready: blocking information missing"
    display(Markdown(f"### {case_id}: {verdict}"))
    display(report.header_table(r.extraction))
    display(report.items_table(r.extraction))
    display(report.issues_table(r))
    if r.client_email:
        display(Markdown("**Draft reply to the client (review before sending):**"))
        print(r.client_email)

**Case 01:** the most complete email. Watch the material: it is not in the email, but the spec reference
is, so the tool points to the spec and archive before suggesting to ask the client.

In [ ]:
show("case01_customer_T")

**Case 02:** watch the quantity check (sets vs. pieces), the missing deadline (urgent wording is not a date),
and whether the material paragraph was read as background or turned into an extra part line.

In [ ]:
show("case02_customer_P")

**Case 03:** Chinese body, abbreviated descriptions, no quantities. The quantities are probably in the zip
attachments, so the tool says to check those before asking the client.

In [ ]:
show("case03_customer_S")

## Step 5 · Scoring against the answer key

- **values caught:** fields the email contains that the extractor got right
- **gaps reported:** fields the email does *not* contain that the extractor correctly left empty
- **invented:** values or whole lines the extractor made up. This is the dangerous kind of mistake.

In [ ]:
def score_all(results):
    return pd.concat([evaluate.score(qt.check(ex, inbox[c]).extraction, evaluate.load_key(KEYS / f"{c}.json"), c)
                      for c, ex in results.items()])

scores = {"baseline (rules)": score_all(baseline_results)}
for model, answers in llm_results.items():
    scores[model] = score_all(answers)

for name, s in scores.items():
    display(Markdown(f"**{name}**"))
    display(evaluate.summary(s))

In [ ]:
# Every field an extractor got wrong
for name, s in scores.items():
    display(Markdown(f"**{name}: mistakes**"))
    display(s[~s["outcome"].str.startswith("correct")][["case", "where", "field", "expected", "predicted", "outcome"]])

## What to take from this

- **Rules work only for layouts we have already seen.** The baseline is close to perfect on Case 01, but it
  turns the material paragraph in Case 02 into an extra part line, and it cannot read 铜 (copper/bronze) or link
  Case 03's lines to the zip files. Every new customer would need new rules.
- **Missing information is the normal case.** Two of three emails have no deadline, currency or incoterm;
  one has no quantities. The tool reports these as gaps instead of guessing, and the checkpoint turns them into
  a reply to the client.
- **The checks catch the extractor's mistakes.** An invented part line shows up as a blocking issue with a
  strange part number, and evidence that is not in the email is downgraded to *check*.
- **For the LLM run, compare:** does it catch the substitute material, the 铜 → bronze reading and the zip
  links, which the rules miss? Does it invent anything? A single *invented* value matters more than a missed one.
- **Limits:** three emails is a demonstration, not a statistical test. The answer key was written by us, and
  some fields (urgency, generic material) are judgement calls; accepted alternatives are marked with `|`.